# MNLP 2026 - Homework 1
## Bi-Encoder Text Retrieval Pipeline
**Authors:** SERRA - FALANGA

This notebook implements a comprehensive Information Retrieval (IR) pipeline based on dense text representations. The objective is to map queries and document chunks into a shared latent vector space to perform semantic search via Cosine Similarity.

In [ ]:
!pip install -q transformers datasets evaluate

In [ ]:
import os
import gc
import json
import random
import collections
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from matplotlib.backends.backend_pdf import PdfPages

# --- GLOBAL CONFIGURATION ---
# Toggle this to True to train models, or False to load existing .pth weights
RUN_TRAINING = True
EPOCHS = 1
BATCH_SIZE = 32
LEARNING_RATE = 2e-5

# Negative Sampling Strategy: "random" or "hard"
NEGATIVE_SAMPLING_STRATEGY = "random"

# Paths
DRIVE_BASE_PATH = '/content/drive/MyDrive/MNLP_HW1_Serra_Falanga'
WEIGHTS_DIR = os.path.join(DRIVE_BASE_PATH, 'Weights_Models')

# Mount Drive
drive.mount('/content/drive')
if os.path.exists(DRIVE_BASE_PATH):
    os.chdir(DRIVE_BASE_PATH)
    print(f"Working directory: {os.getcwd()}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
#AUX Functions
def clear_gpu_memory():
    """Clears Python garbage collector and CUDA cache to free up GPU VRAM."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU Memory Cleared. Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

def mean_pooling(model_output, attention_mask):
    """Perform mean pooling on token embeddings."""
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def get_transformer_embeddings(text_list, tokenizer, model):
    """Get embeddings using a standard AutoModel with mean pooling."""
    if isinstance(text_list, str):
        text_list = [text_list]
    encoded_input = tokenizer(text_list, padding=True, truncation=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        model_output = model(**encoded_input)
    return mean_pooling(model_output, encoded_input['attention_mask'])

## Load Dataset

In [ ]:
ds = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")
train_data = ds['train']
test_data  = ds['test']
blind_data = ds['blind']

print("Dataset loaded successfully:")
print(ds)

# 1. BASELINE EVALUATION
Performance establishment using pre-trained models: `all-MiniLM-L6-v2` and `distilbert-base-uncased` in a zero-shot setting.

## 1.1 Baseline with AutoModel

In [ ]:
print("Loading Baseline Models...")
tokenizer_minilm = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model_minilm_obj = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').to(DEVICE)

tokenizer_distilbert = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model_distilbert_obj = AutoModel.from_pretrained('distilbert-base-uncased').to(DEVICE)

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
def evaluate_transformer_model(data, tokenizer, model, model_name="Model", variant="baseline"):
    """
    Evaluates a transformer model on the provided data and saves ranked results to a .jsonl file.
    """
    hit_counts = {1: 0, 3: 0, 5: 0}
    total_samples = len(data)

    # Normalize model name for filename
    clean_model_name = model_name.lower().replace(" ", "_")
    filename = f"galline_vecchie_fan_buon_brothers-test-{clean_model_name}-{variant}.jsonl"

    print(f"\n--- Starting evaluation for {model_name} on {total_samples} samples ---")
    print(f"Results will be saved to: {filename}")

    model.eval()

    with open(filename, 'w', encoding='utf-8') as f:
        for sample in tqdm(data):
            query = sample['query']
            candidates = sample['candidate_chunks']
            ground_truth_idx = sample['answer_pos']

            # Assuming 'id' or 'query_id' is present in the dataset
            query_id = sample["query_id"]

            # 1. Encode
            with torch.no_grad():
                query_emb = get_transformer_embeddings(query, tokenizer, model)
                cand_embs = get_transformer_embeddings(candidates, tokenizer, model)

            # 2. Similarities
            similarities = F.cosine_similarity(query_emb, cand_embs)

            # 3. Rank and Score
            ranked_indices = torch.argsort(similarities, descending=True).cpu().tolist()

            # 4. Save to JSONL
            # Format: {query_id: [ranked_indices]}
            json_line = json.dumps({query_id: ranked_indices})
            f.write(json_line + "\n")

            for k in hit_counts.keys():
                if ground_truth_idx in ranked_indices[:k]:
                    hit_counts[k] += 1

    # Display Results
    print(f"\n--- {model_name} Final Results ---")
    for k, count in hit_counts.items():
        score = (count / total_samples) * 100
        print(f"Hit@{k}: {score:.2f}% ({count}/{total_samples})")

    return hit_counts

## 1.2 MiniLM with SentenceTransformer

In [ ]:
def evaluate_st_minilm(data, model, variant="baseline"):
    """
    Evaluates MiniLM using the native .encode() method and saves ranked results to a .jsonl file.
    """
    hit_counts = {1: 0, 3: 0, 5: 0}
    total_samples = len(data)

    # Use MiniLM name for filename convention
    filename = f"galline_vecchie_fan_buon_brothers-test-minilm_st-{variant}.jsonl"

    print(f"--- Starting SentenceTransformer Evaluation on {total_samples} samples ---")
    print(f"Results will be saved to: {filename}")

    with open(filename, 'w', encoding='utf-8') as f:
        for sample in tqdm(data):
            query = sample['query']
            candidates = sample['candidate_chunks']
            ground_truth_idx = sample['answer_pos']
            query_id = sample["query_id"]

            # Encode query and candidates using the library's native method
            query_emb = model.encode(query, convert_to_tensor=True)
            cand_embs = model.encode(candidates, convert_to_tensor=True)

            # Compute cosine similarity
            similarities = F.cosine_similarity(query_emb.unsqueeze(0), cand_embs)

            # Ranking
            ranked_indices = torch.argsort(similarities, descending=True).cpu().tolist()

            # Save to JSONL
            json_line = json.dumps({query_id: ranked_indices})
            f.write(json_line + "\n")

            for k in hit_counts.keys():
                if ground_truth_idx in ranked_indices[:k]:
                    hit_counts[k] += 1

    # Results
    print(f"\n--- MiniLM (SentenceTransformer) Results ---")
    for k, count in hit_counts.items():
        score = (count / total_samples) * 100
        print(f"Hit@{k}: {score:.2f}% ({count}/{total_samples})")

    return hit_counts

## 1.3 Baseline Performance Evaluation

In [ ]:
# Run evaluation for DistilBERT
distilbert_results = evaluate_transformer_model(test_data, tokenizer_distilbert, model_distilbert_obj, "DistilBERT")

# Run evaluation for MiniLM
minilm_results = evaluate_transformer_model(test_data, tokenizer_minilm, model_minilm_obj, "MiniLM")

# Run evaluation
minilm_st_results = evaluate_st_minilm(test_data, st_model)

## 1.4 A2 Metrics: MRR Calculation

In [ ]:
def calculate_mrr_from_file(data, jsonl_filename):
    """
    Calculates the Mean Reciprocal Rank (MRR) using the rankings saved in a JSONL file.
    """
    # Create a mapping from query_id to ground_truth_idx (answer_pos)
    gt_map = {sample['query_id']: sample['answer_pos'] for sample in data}

    rr_sum = 0.0
    count = 0

    if not os.path.exists(jsonl_filename):
        print(f"File {jsonl_filename} not found.")
        return 0.0

    with open(jsonl_filename, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            # entry is {query_id: [ranked_indices]}
            for q_id, ranked_indices in entry.items():
                if q_id in gt_map:
                    ground_truth_idx = gt_map[q_id]
                    # Find rank (1-based)
                    try:
                        rank = ranked_indices.index(ground_truth_idx) + 1
                        rr_sum += (1.0 / rank)
                        count += 1
                    except ValueError:
                        # Ground truth not in ranked list (should not happen if all candidates were ranked)
                        continue

    mrr = rr_sum / count if count > 0 else 0.0
    print(f"MRR calculated from {jsonl_filename}: {mrr:.4f}")
    return mrr

In [ ]:
# Execution of A2 MRR Metrics using saved files
print("Evaluating MRR for Baseline Models from saved files:")

# Filenames generated by evaluate_transformer_model and evaluate_st_minilm
file_distilbert = "galline_vecchie_fan_buon_brothers-test-distilbert-baseline.jsonl"
file_minilm = "galline_vecchie_fan_buon_brothers-test-minilm-baseline.jsonl"
file_minilm_st = "galline_vecchie_fan_buon_brothers-test-minilm_st-baseline.jsonl"

mrr_distilbert = calculate_mrr_from_file(test_data, file_distilbert)
mrr_minilm = calculate_mrr_from_file(test_data, file_minilm)
mrr_minilm_st = calculate_mrr_from_file(test_data, file_minilm_st)

print(f"\n--- A2 Metrics: MRR ---")
print(f"DistilBERT MRR:  {mrr_distilbert:.4f}")
print(f"MiniLM MRR:      {mrr_minilm:.4f}")
print(f"MiniLM (ST) MRR: {mrr_minilm_st:.4f}")

# 2. FINE-TUNING
Based on the `RUN_TRAINING` flag, we either fine-tune the models using Triplet Loss and Hard Negative Mining or load the pre-trained weights from Drive.

## 2.1 Supervised Training Loop (B2)

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, data, tokenizer, model=None, max_length=512, negative_strategy="random"):
        """
        Dataset for training with Triplets (Anchor, Positive, Negative).

        Args:
            data: List of examples (e.g., train_data).
            tokenizer: HuggingFace Tokenizer.
            model: Model to compute hard negatives (required only if strategy='hard').
            max_length: Maximum sequence length.
            negative_strategy: "random" or "hard".
        """
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.model = model
        self.negative_strategy = negative_strategy

    def __len__(self):
        return len(self.data)

    def _get_random_negative(self, candidates, pos_idx):
        """Selects a random negative from candidate chunks."""
        neg_indices = [i for i in range(len(candidates)) if i != pos_idx]
        if not neg_indices:
            return candidates[pos_idx] # Extreme fallback
        return candidates[random.choice(neg_indices)]

    def _get_hard_negative(self, anchor_text, candidates, pos_idx):
        """Selects the most similar negative chunk to the query using the current model."""
        neg_indices = [i for i in range(len(candidates)) if i != pos_idx]
        if not neg_indices:
            return candidates[pos_idx]

        neg_texts = [candidates[i] for i in neg_indices]

        self.model.eval()
        with torch.no_grad():
            # Local tokenization for the specific sample candidates
            enc_a = self.tokenizer(anchor_text, return_tensors='pt', truncation=True, padding=True).to(DEVICE)
            enc_n = self.tokenizer(neg_texts, return_tensors='pt', truncation=True, padding=True).to(DEVICE)

            # Embeddings via mean_pooling
            emb_a = mean_pooling(self.model(**enc_a), enc_a['attention_mask'])
            emb_n = mean_pooling(self.model(**enc_n), enc_n['attention_mask'])

            # Similarity between query and all candidate negatives
            similarities = F.cosine_similarity(emb_a, emb_n)
            # Hard negative is the one with the maximum similarity
            hard_idx = torch.argmax(similarities).item()
            return neg_texts[hard_idx]

    def __getitem__(self, idx):
        item = self.data[idx]
        anchor = item['query']
        pos_idx = item['answer_pos']
        positive = item['candidate_chunks'][pos_idx]

        # Strategy routing
        if self.negative_strategy == "hard":
            negative = self._get_hard_negative(anchor, item['candidate_chunks'], pos_idx)
        else:
            # Default to random strategy
            negative = self._get_random_negative(item['candidate_chunks'], pos_idx)

        return anchor, positive, negative

    def collate_fn(self, batch):
        anchors, positives, negatives = zip(*batch)

        def tokenize(texts):
            return self.tokenizer(list(texts), padding=True, truncation=True,
                                  max_length=self.max_length, return_tensors='pt').to(DEVICE)

        return tokenize(anchors), tokenize(positives), tokenize(negatives)

def train_triplet_epoch(model, dataloader, optimizer, margin=0.5):
    """Performs one training epoch using Triplet Margin Loss."""
    model.train()
    total_loss = 0
    # Using TripletMarginLoss which is standard for bi-encoders
    criterion = nn.TripletMarginLoss(margin=margin, p=2)

    for batch in tqdm(dataloader, desc="Training", leave=False):
        anchors, positives, negatives = batch
        optimizer.zero_grad()

        # Compute embeddings for all three parts of the triplet
        emb_a = mean_pooling(model(**anchors), anchors['attention_mask'])
        emb_p = mean_pooling(model(**positives), positives['attention_mask'])
        emb_n = mean_pooling(model(**negatives), negatives['attention_mask'])

        # Loss calculation
        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

def fine_tune_model(model, train_data, tokenizer, strategy="random", epochs=1, batch_size=32, lr=2e-5):
    """Fine-tuning wrapper with strategy selection."""
    print(f"\n>>> Starting Fine-Tuning | Strategy: {strategy.upper()} | Epochs: {epochs}")

    # Initialize dataset with the model passed explicitly regardless of strategy
    dataset = TripletDataset(train_data, tokenizer, model=model, negative_strategy=strategy)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=dataset.collate_fn)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        avg_loss = train_triplet_epoch(model, dataloader, optimizer)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

    return model

In [ ]:
if RUN_TRAINING:
    print(f"Training mode enabled. Starting Fine-Tuning for {EPOCHS} epochs...")

    # Fine-tune DistilBERT using the global NEGATIVE_SAMPLING_STRATEGY
    print("\n--- Fine-tuning DistilBERT ---")
    model_distilbert_ft = fine_tune_model(
        model_distilbert_obj,
        train_data,
        tokenizer_distilbert,
        strategy=NEGATIVE_SAMPLING_STRATEGY,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE
    )
    # Save to Weights_Models
    torch.save(model_distilbert_ft.state_dict(), os.path.join(WEIGHTS_DIR, 'distilbert_ft_weights.pth'))
    print(f"DistilBERT weights saved to {WEIGHTS_DIR}")

    # Fine-tune MiniLM using the global NEGATIVE_SAMPLING_STRATEGY
    print("\n--- Fine-tuning MiniLM ---")
    model_minilm_ft = fine_tune_model(
        model_minilm_obj,
        train_data,
        tokenizer_minilm,
        strategy=NEGATIVE_SAMPLING_STRATEGY,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE
    )
    # Save to Weights_Models
    torch.save(model_minilm_ft.state_dict(), os.path.join(WEIGHTS_DIR, 'minilm_ft_weights.pth'))
    print(f"MiniLM weights saved to {WEIGHTS_DIR}")

else:
    print("Loading mode enabled. Fetching weights from Drive/Local directory...")

    # Load DistilBERT
    model_distilbert_ft = AutoModel.from_pretrained('distilbert-base-uncased').to(DEVICE)
    dist_path = os.path.join(WEIGHTS_DIR, 'distilbert_ft_weights.pth')
    if os.path.exists(dist_path):
        model_distilbert_ft.load_state_dict(torch.load(dist_path, map_location=DEVICE))
        print(f"DistilBERT loaded from {dist_path}")
    else:
        print("Warning: DistilBERT fine-tuned weights not found. Using base model.")

    # Load MiniLM
    model_minilm_ft = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').to(DEVICE)
    mini_path = os.path.join(WEIGHTS_DIR, 'minilm_ft_weights.pth')
    if os.path.exists(mini_path):
        model_minilm_ft.load_state_dict(torch.load(mini_path, map_location=DEVICE))
        print(f"MiniLM loaded from {mini_path}")
    else:
        print("Warning: MiniLM fine-tuned weights not found. Using base model.")

model_distilbert_ft.eval()
model_minilm_ft.eval()
print("\nModels are ready for evaluation.")

## 2.2 Evaluation of Fine-Tuned Models (A2)

In [ ]:
# Evaluate Fine-tuned DistilBERT using the model already loaded in the previous cell
print("Evaluating Fine-tuned DistilBERT...")
distilbert_ft_results = evaluate_transformer_model(test_data, tokenizer_distilbert, model_distilbert_ft, "DistilBERT", variant="fine_tuned")

# Evaluate Fine-tuned MiniLM using the model already loaded in the previous cell
print("\nEvaluating Fine-tuned MiniLM...")
minilm_ft_results = evaluate_transformer_model(test_data, tokenizer_minilm, model_minilm_ft, "MiniLM", variant="fine_tuned")

# Define filenames for MRR calculation
file_distilbert_ft = "galline_vecchie_fan_buon_brothers-test-distilbert-fine_tuned.jsonl"
file_minilm_ft = "galline_vecchie_fan_buon_brothers-test-minilm-fine_tuned.jsonl"

print("\n--- A2 Metrics: MRR for Fine-tuned Models ---")
mrr_distilbert_ft = calculate_mrr_from_file(test_data, file_distilbert_ft)
mrr_minilm_ft = calculate_mrr_from_file(test_data, file_minilm_ft)

# Final Comparison Summary
print("\n--- SUMMARY COMPARISON (Fine-tuned) ---")
print(f"DistilBERT (FT) - Hit@1: {distilbert_ft_results[1]/len(test_data)*100:.2f}% | MRR: {mrr_distilbert_ft:.4f}")
print(f"MiniLM (FT)     - Hit@1: {minilm_ft_results[1]/len(test_data)*100:.2f}% | MRR: {mrr_minilm_ft:.4f}")

# 3. ADVANCED ARCHITECTURE (A4)
The A4 phase focuses on architectural experimentation and stability improvements.

### Implementation Details:
- **Model Selection**: Integration of a custom candidate (e.g., TinyBERT or MPNet) to evaluate performance scaling.
- **Comparative Analysis**: Executing a systematic Pre-vs-Post Fine-Tuning evaluation to validate learning gains.
- **Resource Management**: Implementation of CUDA memory management and garbage collection routines to ensure training stability across different model sizes.

## 3.1 GPU Resource Management & Model Offloading

In [ ]:
# 3.1 GPU Resource Management & Model Offloading
if RUN_TRAINING:
    print("Training mode active: Cleaning up GPU memory for next stage...")

    # Delete heavy models from GPU memory
    models_to_delete = ['model_distilbert_obj', 'model_distilbert_ft', 'model_minilm_obj', 'model_minilm_ft']

    for m in models_to_delete:
        if m in locals():
            del locals()[m]

    # Deep cleanup
    gc.collect()
    torch.cuda.empty_cache()

    print("\nGPU Memory Cleared!")
    print(f"Allocated Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
else:
    print("Loading mode active: Skipping GPU cleanup to keep baseline models available.")

## 3.2 Advanced Architecture Loading (TinyBERT)

In [ ]:
# Load TinyBERT as a candidate for A4
model_a4_name = 'huawei-noah/TinyBERT_General_4L_312D'
print(f"Loading {model_a4_name} for A4...")

tokenizer_a4 = AutoTokenizer.from_pretrained(model_a4_name)
model_a4 = AutoModel.from_pretrained(model_a4_name).to(DEVICE)

## 3.3 Zero-Shot Evaluation (A4 Pre-FT)

In [ ]:
print("--- Evaluation A4 (TinyBERT): ZERO-SHOT ---")
# 1. Load Base Model
model_a4_name = 'huawei-noah/TinyBERT_General_4L_312D'
tokenizer_a4 = AutoTokenizer.from_pretrained(model_a4_name)
model_a4_base = AutoModel.from_pretrained(model_a4_name).to(DEVICE)

# 2. Pre-FT Evaluation (now labeled baseline)
results_a4_pre = evaluate_transformer_model(test_data, tokenizer_a4, model_a4_base, "TinyBERT", variant="baseline")

file_a4_pre = "galline_vecchie_fan_buon_brothers-test-tinybert-baseline.jsonl"
mrr_a4_pre = calculate_mrr_from_file(test_data, file_a4_pre)

## 3.4 Domain-Specific Fine-Tuning (A4)

In [ ]:
if RUN_TRAINING:
    print(f"--- Fine-Tuning A4 (TinyBERT) | Strategy: {NEGATIVE_SAMPLING_STRATEGY.upper()} ---")

    # Fine-tune TinyBERT using the same logic as baseline models
    model_a4_ft = fine_tune_model(
        model_a4_base,
        train_data,
        tokenizer_a4,
        strategy=NEGATIVE_SAMPLING_STRATEGY,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE
    )

    # Save fine-tuned weights to Weights_Models
    tinybert_weights_path = os.path.join(WEIGHTS_DIR, 'tinybert_ft_weights.pth')
    torch.save(model_a4_ft.state_dict(), tinybert_weights_path)
    print(f"TinyBERT weights saved to {tinybert_weights_path}")

else:
    print("Loading mode enabled for A4. Fetching weights from Drive...")
    model_a4_ft = AutoModel.from_pretrained(model_a4_name).to(DEVICE)

    tinybert_ft_path = os.path.join(WEIGHTS_DIR, 'tinybert_ft_weights.pth')

    if os.path.exists(tinybert_ft_path):
        model_a4_ft.load_state_dict(torch.load(tinybert_ft_path, map_location=DEVICE))
        print(f"TinyBERT fine-tuned weights loaded from {tinybert_ft_path}")
    else:
        print("Warning: Fine-tuned weights not found. Using base TinyBERT model.")

model_a4_ft.eval()
print("TinyBERT (A4) is ready for post-FT evaluation.")

### 3.5 Post-Training Evaluation (A4 Results)

After obtaining the optimized TinyBERT (either through training or loading), we evaluate its performance on the test set to compare it with the baseline.

In [ ]:
print("--- Evaluation A4 (TinyBERT): POST FINE-TUNING ---")
results_a4_post = evaluate_transformer_model(test_data, tokenizer_a4, model_a4_ft, "TinyBERT", variant="fine_tuned")

file_a4_post = "galline_vecchie_fan_buon_brothers-test-tinybert-fine_tuned.jsonl"
mrr_a4_post = calculate_mrr_from_file(test_data, file_a4_post)

# 4. BLIND TEST INFERENCE
The final stage involves deploying the optimized fine-tuned models on the unlabeled `blind` dataset.

### Deliverables:
Generation of `.jsonl` files containing the ranked candidate indices for each query. These outputs represent the final submission for the retrieval task, reflecting the cumulative improvements from baseline and fine-tuning stages.

In [ ]:
# 4.1 Reload Fine-tuned Models for Blind Inference
print("Reloading fine-tuned models for blind test...")

# Reload DistilBERT FT
model_distilbert_ft = AutoModel.from_pretrained('distilbert-base-uncased').to(DEVICE)
dist_path = os.path.join(WEIGHTS_DIR, 'distilbert_ft_weights.pth')
if os.path.exists(dist_path):
    model_distilbert_ft.load_state_dict(torch.load(dist_path, map_location=DEVICE))
    print(f"DistilBERT FT reloaded from {dist_path}")

# Reload MiniLM FT
model_minilm_ft = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2').to(DEVICE)
mini_path = os.path.join(WEIGHTS_DIR, 'minilm_ft_weights.pth')
if os.path.exists(mini_path):
    model_minilm_ft.load_state_dict(torch.load(mini_path, map_location=DEVICE))
    print(f"MiniLM FT reloaded from {mini_path}")

model_distilbert_ft.eval()
model_minilm_ft.eval()
print("Models successfully reloaded and set to evaluation mode.")

In [ ]:
def generate_blind_results(data, tokenizer, model, model_label):
    """Generates the JSONL file for the blind test without calculating metrics (no ground truth available)"""
    filename = f"galline_vecchie_fan_buon_brothers-blind-{model_label}.jsonl"
    print(f"Generating blind results for {model_label} -> {filename}")

    model.eval()
    with open(filename, 'w', encoding='utf-8') as f:
        for sample in tqdm(data):
            query = sample['query']
            candidates = sample['candidate_chunks']
            query_id = sample["query_id"]

            with torch.no_grad():
                query_emb = get_transformer_embeddings(query, tokenizer, model)
                cand_embs = get_transformer_embeddings(candidates, tokenizer, model)

            similarities = F.cosine_similarity(query_emb, cand_embs)
            ranked_indices = torch.argsort(similarities, descending=True).cpu().tolist()

            f.write(json.dumps({query_id: ranked_indices}) + "\n")

# Execute Blind Test on all optimized models
generate_blind_results(blind_data, tokenizer_distilbert, model_distilbert_ft, "distilbert_ft")
generate_blind_results(blind_data, tokenizer_minilm, model_minilm_ft, "minilm_ft")
generate_blind_results(blind_data, tokenizer_a4, model_a4_ft, "tinybert_ft")

# 5. COMPARATIVE ANALYSIS & STATISTICS
This section provides a quantitative comparison between the baseline models and their fine-tuned counterparts. We analyze the performance gains achieved through supervised triplet training and hard negative mining.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# 1. Aggregate results into a structured format
results_map = {
    ('DistilBERT', 'Baseline'): (distilbert_results if 'distilbert_results' in locals() else None, mrr_distilbert if 'mrr_distilbert' in locals() else 0),
    ('DistilBERT', 'Fine-Tuned'): (distilbert_ft_results if 'distilbert_ft_results' in locals() else None, mrr_distilbert_ft if 'mrr_distilbert_ft' in locals() else 0),
    ('MiniLM', 'Baseline'): (minilm_results if 'minilm_results' in locals() else None, mrr_minilm if 'mrr_minilm' in locals() else 0),
    ('MiniLM', 'Fine-Tuned'): (minilm_ft_results if 'minilm_ft_results' in locals() else None, mrr_minilm_ft if 'mrr_minilm_ft' in locals() else 0),
    ('TinyBERT (A4)', 'Baseline'): (results_a4_pre if 'results_a4_pre' in locals() else None, mrr_a4_pre if 'mrr_a4_pre' in locals() else 0),
    ('TinyBERT (A4)', 'Fine-Tuned'): (results_a4_post if 'results_a4_post' in locals() else None, mrr_a4_post if 'mrr_a4_post' in locals() else 0)
}

rows = []
for (model, status), (hit_data, mrr_val) in results_map.items():
    hit1_pc = (hit_data[1] / len(test_data) * 100) if hit_data is not None else 0
    rows.append({
        'Model': model,
        'Status': status,
        'Hit@1 (%)': hit1_pc,
        'MRR': mrr_val
    })

df_stats = pd.DataFrame(rows)

# 2. Visual Comparison (Hit@1 and MRR)
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

# Plot Hit@1
sns.barplot(data=df_stats, x='Model', y='Hit@1 (%)', hue='Status', ax=ax1, palette='viridis')
ax1.set_title('Hit@1 Comparison: Baseline vs Fine-Tuned', fontsize=14, fontweight='bold')
ax1.set_ylabel('Hit@1 (%)')

# Plot MRR
sns.barplot(data=df_stats, x='Model', y='MRR', hue='Status', ax=ax2, palette='magma')
ax2.set_title('MRR Comparison: Baseline vs Fine-Tuned', fontsize=14, fontweight='bold')
ax2.set_ylabel('MRR Score')

plt.tight_layout()
plt.show()

# 3. Save only the bar charts to PDF
pdf_filename = "model_performance_report.pdf"
with PdfPages(pdf_filename) as pdf:
    pdf.savefig(fig)

print(f"\nStatistical report (bar charts only) saved as: {pdf_filename}")